In [19]:
import pandas as pd
from collections import Counter
import numpy as np
from pathlib import Path
from scipy.stats import entropy  # entropy is a function to compute KL-divergence


In [20]:
#Define QID - question matching for overlapping questions -- ANES 2024
qids = ["V241383", "V242250", "V242276", "V242321", 
        "V242325", "V242332", "V242347", "V242351"]
name_dict = {"V241383" : "Gay Marriage", 
            "V242250" : "Income Inequality", 
            "V242276" : "Gender Role", "V242321": "Climate Change",
            "V242325": "Gun Regulation", "V242332": "Drug Addiction", "V242347": "Race Diversity", "V242351": "Health Insurance"}

def get_counts(path, qids): #function to get a dict with counts for each question
    value_list = []

    for id in qids:
        df = pd.read_csv(path + id + ".csv")
        df['Response'] = pd.to_numeric(df['Response'], errors='coerce') 
        value_counts = Counter(df['Response'].dropna())

        if id in ["V242321"]:
            answers = {i: value_counts[i] for i in range(1, 6)}
        else:
            answers = {i: value_counts[i] for i in range(1, 4)}

        value_list.append({name_dict[id]: answers})

    return value_list


### Select model

In [21]:
# GPT-4.1
model_name = 'gpt4.1'
 # T=0
replicate_path = "../Publication Results/Gpt4.1-Mini-Determinstic/main_mq_results_2024/full_results_2024_"
reformulated_path = "../Publication Results/Gpt4.1-Mini-Determinstic/main_mq_reform_results_2024/full_results_2024_"

replicate_results = get_counts(replicate_path, qids)
reformulated_results = get_counts(reformulated_path, qids)


In [22]:
def get_human_counts(path, qids): #getting results from anes 2020
     anes_df = pd.read_csv(path)
     human_values = []

     for id in qids:
          human_counts = Counter(anes_df[id].dropna())
          if id in ["V242321"]:
               answers = {i: human_counts[i] for i in range(1, 6)}
          else:
               answers = {i: human_counts[i] for i in range(1, 4)}

          human_values.append({name_dict[id]: answers})

     return human_values

human_results = get_human_counts("../data/2024 ANES_test.csv", qids)

/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_56464/77143347.py:2: DtypeWarning: Columns (8,10,11,16,18,19,20,22,23,24,26,27,28,30,31,36,38,39,1448,1449,1450,1451,1452,1455,1456,1457,1458,1459,1463,1464,1465) have mixed types. Specify dtype option on import or set low_memory=False.
  anes_df = pd.read_csv(path)


In [23]:
#JSD divergence is a more numerically stable version of KL-divergence - better for our case
def jsd_divergence_between_sets(set1, set2, base=2):
    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq

    return jsd_results

In [24]:
#Get all the results and divergences
jsd_results_replicate = jsd_divergence_between_sets(human_results, replicate_results, base=2)
jsd_results_reformulated = jsd_divergence_between_sets(human_results, reformulated_results, base=2)

diff_reformulated_vs_replicate = {key: (jsd_results_reformulated[key] - jsd_results_replicate[key]) for key in jsd_results_replicate.keys()}


In [25]:
# format data
# Difference in JS-divergence between replicate and 4 experimental conditions
results_mapping = [
    ('Replicate', jsd_results_replicate),
    ('Reformulated - Replicate', diff_reformulated_vs_replicate),
]

question_order = ['Climate Change','Health Insurance','Income Inequality','Race Diversity','Drug Addiction','Gay Marriage','Gun Regulation','Gender Role']
sig_df = pd.DataFrame({'Question': question_order})
for column_name, result_dict in results_mapping:
    ordered_values = [
        result_dict.get(q, np.nan)  # Use .get() and np.nan for missing keys
        for q in question_order
    ]
    sig_df[column_name] = np.round(ordered_values, 3)

In [26]:
print(model_name)
sig_df

gpt4.1


,Question,Replicate,Reformulated - Replicate
0,Climate Change,0.086,-0.050
1,Health Insurance,0.029,0.004
2,Income Inequality,0.023,0.087
3,Race Diversity,0.293,-0.110
4,Drug Addiction,0.123,-0.005
5,Gay Marriage,0.032,-0.026
6,Gun Regulation,0.001,0.016
7,Gender Role,0.028,-0.012


In [27]:
def jsd_divergence_for_bootstrapping(human_sample, bootstrap_sample, base=2):
    p_counts = human_sample
    q_counts = bootstrap_sample

    p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
    q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

    p /= p.sum()
    q /= q.sum()

    epsilon = 1e-10
    p = np.clip(p, epsilon, 1)
    q = np.clip(q, epsilon, 1)

    m = 0.5 * (p + q)
    jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base))

    return jsd_pq


### CIs (Human vs conditions)

In [28]:
human_df = pd.read_csv("../data/2024 ANES_test.csv")

def bootstrap_jsd(path, qid, n_boot=2000, base=2, return_samples=False):

    question_df = pd.read_csv(path + qid + ".csv")
    question_df['Response'] = pd.to_numeric(question_df['Response'], errors='coerce')

    responses = np.array(question_df['Response'].dropna())

    human_responses = {list(d.keys())[0]: list(d.values())[0] for d in human_results}[name_dict[qid]]

    jsd_samples = []

    for _ in range(n_boot):
        sample = np.random.choice(responses, size=len(responses), replace=True) #sample with replacement
        
        counts = Counter(sample)

        jsd_val = jsd_divergence_for_bootstrapping(human_responses, counts, base=base) #compute JSD for this iteration
        jsd_samples.append(jsd_val) #append to the samples list

    jsd_samples = np.array(jsd_samples)

    if return_samples:
        return jsd_samples  # <— raw samples for delta CI

    mean_jsd = jsd_samples.mean()
    ci_lower, ci_upper = np.percentile(jsd_samples, [2.5, 97.5])
    
    return (("mean", mean_jsd), ("ci_lower", ci_lower), ("ci_upper", ci_upper))
 

/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_56464/2242348685.py:1: DtypeWarning: Columns (8,10,11,16,18,19,20,22,23,24,26,27,28,30,31,36,38,39,1448,1449,1450,1451,1452,1455,1456,1457,1458,1459,1463,1464,1465) have mixed types. Specify dtype option on import or set low_memory=False.
  human_df = pd.read_csv("../data/2024 ANES_test.csv")


In [29]:
#Create CIs (human vs conditions) and compare them
ci_replicate = []
ci_reformulated = []

for qid in qids:
    ci_replicate.append((name_dict[qid], bootstrap_jsd(path = replicate_path, qid = qid)))
    ci_reformulated.append((name_dict[qid], bootstrap_jsd(path = reformulated_path, qid = qid)))

ci_replicate = [{item[0] : item[1]} for item in ci_replicate]
ci_reformulated = [{item[0] : item[1]} for item in ci_reformulated]

ci_replicate = {list(d.keys())[0]: list(d.values())[0] for d in ci_replicate}
ci_reformulated = {list(d.keys())[0]: list(d.values())[0] for d in ci_reformulated}


In [30]:
# Mean JS divergence and 95% bootstrap confidence intervals between ANSE and silicon distributions by item and condition.
def format_ci_data(ci_tuple):
    # Convert the list of tuples structure into a dictionary for easy access
    if ci_tuple is None:
        return "N/A"
    data = dict(ci_tuple)
    
    mean = data['mean']
    ci_lower = data['ci_lower']
    ci_upper = data['ci_upper']

    # Round and format
    mean_str = f"{mean:.4f}"
    lower_str = f"{ci_lower:4f}"
    upper_str = f"{ci_upper:.4f}"

    return f"{mean_str} [{lower_str}, {upper_str}]"

results_mapping = [
    ('Replicate', ci_replicate),
    ('Reformulated', ci_reformulated),
]
question_order = ['Climate Change','Health Insurance','Income Inequality','Race Diversity','Drug Addiction','Gay Marriage','Gun Regulation','Gender Role']

ci_df = pd.DataFrame({'Question': question_order})
for column_name, result_dict in results_mapping:
    ordered_formatted_values = [
        format_ci_data(result_dict.get(q,None))
        for q in question_order
    ]

    ci_df[column_name] = ordered_formatted_values


In [31]:
ci_df

,Question,Replicate,Reformulated
0,Climate Change,"0.0866 [0.083193, 0.0904]","0.0364 [0.032645, 0.0403]"
1,Health Insurance,"0.0290 [0.025320, 0.0329]","0.0330 [0.029299, 0.0370]"
2,Income Inequality,"0.0234 [0.020258, 0.0269]","0.1100 [0.104475, 0.1154]"
3,Race Diversity,"0.2931 [0.286118, 0.3007]","0.1827 [0.175187, 0.1902]"
4,Drug Addiction,"0.1229 [0.120123, 0.1256]","0.1176 [0.116241, 0.1189]"
5,Gay Marriage,"0.0317 [0.028010, 0.0357]","0.0056 [0.003990, 0.0073]"
6,Gun Regulation,"0.0013 [0.000624, 0.0021]","0.0172 [0.014329, 0.0201]"
7,Gender Role,"0.0286 [0.025427, 0.0317]","0.0162 [0.013538, 0.0191]"
